In [ ]:
!pip install -q transformers datasets evaluate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import random
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
dataset = load_dataset("imdb")

train_dataset = dataset["train"].shuffle(seed=SEED).select(range(10000))
test_dataset = dataset["test"].shuffle(seed=SEED).select(range(2000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels)
    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
baseline_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Freeze transformer layers
for param in baseline_model.distilbert.parameters():
    param.requires_grad = False

baseline_model.to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,

    # ✅ NEW name in your transformers version
    eval_strategy="epoch",

    logging_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
)

In [ ]:
baseline_trainer = Trainer(
    model=baseline_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

baseline_trainer.train()
baseline_results = baseline_trainer.evaluate()
print("Baseline Results:", baseline_results)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.633122,0.568747,0.798500,0.792802
2,0.555790,0.533301,0.806500,0.809453


Baseline Results: {'eval_loss': 0.5333009362220764, 'eval_accuracy': 0.8065, 'eval_f1': 0.8094534711964549, 'eval_runtime': 6.4631, 'eval_samples_per_second': 309.449, 'eval_steps_per_second': 19.341, 'epoch': 2.0}


In [ ]:
full_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

full_model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
full_trainer = Trainer(
    model=full_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

full_trainer.train()
full_results = full_trainer.evaluate()
print("Full Fine-Tuning Results:", full_results)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.339629,0.252286,0.900000,0.898580
2,0.192454,0.257428,0.908500,0.908179


Full Fine-Tuning Results: {'eval_loss': 0.25742846727371216, 'eval_accuracy': 0.9085, 'eval_f1': 0.9081786251881585, 'eval_runtime': 6.5607, 'eval_samples_per_second': 304.847, 'eval_steps_per_second': 19.053, 'epoch': 2.0}


In [ ]:
lora_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

lora_model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

In [ ]:
lora_trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

lora_trainer.train()
lora_results = lora_trainer.evaluate()
print("LoRA Results:", lora_results)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.498179,0.334876,0.850500,0.848608
2,0.345610,0.326036,0.857500,0.857997


LoRA Results: {'eval_loss': 0.32603591680526733, 'eval_accuracy': 0.8575, 'eval_f1': 0.8579970104633782, 'eval_runtime': 6.8366, 'eval_samples_per_second': 292.544, 'eval_steps_per_second': 18.284, 'epoch': 2.0}


In [ ]:
print("\n===== FINAL RESULTS =====")
print("Baseline:", baseline_results)
print("Full Fine-Tuning:", full_results)
print("LoRA (PEFT):", lora_results)


===== FINAL RESULTS =====
Baseline: {'eval_loss': 0.5333009362220764, 'eval_accuracy': 0.8065, 'eval_f1': 0.8094534711964549, 'eval_runtime': 6.4631, 'eval_samples_per_second': 309.449, 'eval_steps_per_second': 19.341, 'epoch': 2.0}
Full Fine-Tuning: {'eval_loss': 0.25742846727371216, 'eval_accuracy': 0.9085, 'eval_f1': 0.9081786251881585, 'eval_runtime': 6.5607, 'eval_samples_per_second': 304.847, 'eval_steps_per_second': 19.053, 'epoch': 2.0}
LoRA (PEFT): {'eval_loss': 0.32603591680526733, 'eval_accuracy': 0.8575, 'eval_f1': 0.8579970104633782, 'eval_runtime': 6.8366, 'eval_samples_per_second': 292.544, 'eval_steps_per_second': 18.284, 'epoch': 2.0}


In [ ]:
raw_test_dataset = load_dataset("imdb")["test"].shuffle(seed=42).select(range(2000))

In [ ]:
def get_predictions(trainer, tokenized_dataset, raw_dataset, n=5):
    outputs = trainer.predict(tokenized_dataset)
    preds = np.argmax(outputs.predictions, axis=-1)
    labels = outputs.label_ids

    for i in range(n):
        print("TEXT:", raw_dataset[i]["text"][:300])
        print("TRUE:", labels[i], "PRED:", preds[i])
        print("-"*70)

In [ ]:
print("Baseline examples:")
get_predictions(baseline_trainer, test_dataset, raw_test_dataset)

print("LoRA examples:")
get_predictions(lora_trainer, test_dataset, raw_test_dataset)

print("Full FT examples:")
get_predictions(full_trainer, test_dataset, raw_test_dataset)

Baseline examples:


TEXT: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I sta
TRUE: 1 PRED: 1
----------------------------------------------------------------------
TEXT: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawned at least eight films (none of which was ever released in the U.S.). 'O.S.S.117:Cairo,Nest Of Spies' is a breezy little co
TRUE: 1 PRED: 1
----------------------------------------------------------------------
TEXT: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good time. I at least thought I'd be able to stand it. But, I was wrong. First, the weird looping? It was like

TEXT: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I sta
TRUE: 1 PRED: 1
----------------------------------------------------------------------
TEXT: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawned at least eight films (none of which was ever released in the U.S.). 'O.S.S.117:Cairo,Nest Of Spies' is a breezy little co
TRUE: 1 PRED: 1
----------------------------------------------------------------------
TEXT: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good time. I at least thought I'd be able to stand it. But, I was wrong. First, the weird looping? It was like

TEXT: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I sta
TRUE: 1 PRED: 1
----------------------------------------------------------------------
TEXT: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawned at least eight films (none of which was ever released in the U.S.). 'O.S.S.117:Cairo,Nest Of Spies' is a breezy little co
TRUE: 1 PRED: 1
----------------------------------------------------------------------
TEXT: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good time. I at least thought I'd be able to stand it. But, I was wrong. First, the weird looping? It was like

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ===== Results (from your FINAL RESULTS) =====
models = ["Baseline\n(Frozen)", "LoRA\n(PEFT)", "Full FT\n(FFT)"]

accuracy = np.array([0.8065, 0.8575, 0.9085])
f1 = np.array([0.8094534711964549, 0.8579970104633782, 0.9081786251881585])
loss = np.array([0.5333009362220764, 0.32603591680526733, 0.25742846727371216])

# NOTE: replace these with the EXACT trainable param counts if you printed them.
# Baseline (classifier head only) ~0.8M is typical; Full FT ~66M for DistilBERT.
trainable_m = np.array([0.8, 2.0, 66.0])  # Millions

# -------------------------
# Fig 1: Accuracy & F1 comparison
# -------------------------
x = np.arange(len(models))
width = 0.35
plt.figure(figsize=(8,5))
plt.bar(x - width/2, accuracy, width, label="Accuracy")
plt.bar(x + width/2, f1, width, label="F1")
plt.xticks(x, models)
plt.ylim(0.75, 0.95)
plt.ylabel("Score")
plt.title("IMDB Performance Comparison (Accuracy & F1)")
plt.legend()
plt.tight_layout()
plt.savefig("fig1_accuracy_f1.png", dpi=300)
plt.close()

# -------------------------
# Fig 2: Evaluation loss
# -------------------------
plt.figure(figsize=(7,5))
plt.bar(models, loss)
plt.ylabel("Eval Loss")
plt.title("Evaluation Loss Across Training Regimes")
plt.tight_layout()
plt.savefig("fig2_eval_loss.png", dpi=300)
plt.close()

# -------------------------
# Fig 3: Trainable parameter efficiency (log scale)
# -------------------------
plt.figure(figsize=(7,5))
plt.bar(models, trainable_m)
plt.yscale("log")
plt.ylabel("Trainable Parameters (Millions, log scale)")
plt.title("Parameter Efficiency: Trainable Parameters by Method")
plt.tight_layout()
plt.savefig("fig3_trainable_params_log.png", dpi=300)
plt.close()

# -------------------------
# Fig 4: Efficiency–performance trade-off (log x)
# -------------------------
plt.figure(figsize=(7,5))
plt.scatter(trainable_m, accuracy)
for i, m in enumerate(models):
    plt.annotate(m.replace("\n", " "), (trainable_m[i], accuracy[i]),
                 textcoords="offset points", xytext=(6,6))
plt.xscale("log")
plt.xlabel("Trainable Parameters (Millions, log scale)")
plt.ylabel("Accuracy")
plt.title("Efficiency–Performance Trade-off")
plt.tight_layout()
plt.savefig("fig4_efficiency_tradeoff.png", dpi=300)
plt.close()

print("Saved figures:")
print("fig1_accuracy_f1.png")
print("fig2_eval_loss.png")
print("fig3_trainable_params_log.png")
print("fig4_efficiency_tradeoff.png")

Saved figures:
fig1_accuracy_f1.png
fig2_eval_loss.png
fig3_trainable_params_log.png
fig4_efficiency_tradeoff.png
